# GBR Source Summary – CLI Notebook Examples

This notebook runs the package through the production entry point:

```text
scripts/run_report_card_summary.py
```

This is preferred for production-style runs because it uses the same code path as the command line.

The notebook provides examples for:

- Full report-card bundle
- Fast summary run
- Change summary plots only
- RSDR exports only
- Modelled-vs-measured / Moriasi outputs
- Debugging with `--fail-fast`

Fine sediment should be specified as `FS`, not `TSS`.

## 1. Setup

In [1]:
from pathlib import Path
import subprocess
import sys

# ------------------------------------------------------------------------------
# Locate the repository root
# ------------------------------------------------------------------------------
#
# This notebook is normally stored in:
#
#     GBR_source_summary/example/
#
# but the production CLI script is stored in:
#
#     GBR_source_summary/scripts/run_report_card_summary.py
#
# Therefore we search upwards from the current notebook folder until we find the
# repository folder containing the scripts directory. This makes the notebook work
# whether it is opened from example/, notebooks/, or the repository root.
# ------------------------------------------------------------------------------

def find_repo_root(start: Path) -> Path:
    """Find the nearest parent folder containing scripts/run_report_card_summary.py."""
    start = start.resolve()

    for folder in [start, *start.parents]:
        script = folder / "scripts" / "run_report_card_summary.py"

        if script.exists():
            return folder

    raise FileNotFoundError(
        "Could not find scripts/run_report_card_summary.py. "
        "Open this notebook from inside the GBR_source_summary repository, "
        "or set REPO_ROOT manually."
    )


REPO_ROOT = find_repo_root(Path.cwd())

# Full path to the production report-card runner.
SCRIPT = REPO_ROOT / "scripts" / "run_report_card_summary.py"

print("Python:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)
print("SCRIPT:", SCRIPT)
print("SCRIPT exists:", SCRIPT.exists())

Python: c:\Users\lius\OneDrive - ITP (Queensland Government)\ShuciL\CY_rebuild\Code\GBR_source_summary\.venv\Scripts\python.exe
REPO_ROOT: C:\Users\lius\OneDrive - ITP (Queensland Government)\ShuciL\CY_rebuild\Code\GBR_source_summary
SCRIPT: C:\Users\lius\OneDrive - ITP (Queensland Government)\ShuciL\CY_rebuild\Code\GBR_source_summary\scripts\run_report_card_summary.py
SCRIPT exists: True


## 2. User settings

In [2]:
# ------------------------------------------------------------------------------
# User settings
# ------------------------------------------------------------------------------
#
# Change these values for each report-card run.
#
# MAIN_PATH
#   Root folder for the model/report-card data.
#   This folder should contain the model results folder specified below.
#
# MODEL_RESULTS_PREFIX
#   Folder containing the region-level model outputs.
#   This can be a relative folder under MAIN_PATH, for example:
#
#       F:\RC13_RC2023_24_Gold\Gold_ResultsSets_PointOfTruth
#
#   The CLI resolves this as:
#
#       MAIN_PATH / MODEL_RESULTS_PREFIX
#
# OUTPUT_ROOT
#   Folder where the report-card bundle outputs will be written.
#   Use a different output folder for each test run so previous outputs are not
#   overwritten accidentally.
#
# REGIONS
#   Comma-separated GBR regions to process.
#
# CONSTITUENTS
#   Comma-separated constituents.
#   Use FS for fine sediment. Do not use TSS.
#
# UNITS
#   "report" gives report-card units:
#       FS/CS = kt/yr
#       nutrients = t/yr
#
# PROCESS_MODEL
#   Scenario used by process summaries and plots.
#   Usually BASE for report-card reporting.
# ------------------------------------------------------------------------------

MAIN_PATH = r"F:\RC13_RC2023_24_Gold"

MODEL_RESULTS_PREFIX = "Gold_ResultsSets_PointOfTruth"

OUTPUT_ROOT = r"F:\RC13_RC2023_24_Gold\report_card_bundle_notebook"

REGIONS = "CY,WT,MW,FI,BU,BM"

CONSTITUENTS = "FS,PN,PP,DIN,DON,DIP,DOP"

UNITS = "report"

PROCESS_MODEL = "BASE"

print("MAIN_PATH:", MAIN_PATH)
print("MODEL_RESULTS_PREFIX:", MODEL_RESULTS_PREFIX)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("REGIONS:", REGIONS)
print("CONSTITUENTS:", CONSTITUENTS)
print("UNITS:", UNITS)
print("PROCESS_MODEL:", PROCESS_MODEL)

MAIN_PATH: F:\RC13_RC2023_24_Gold
MODEL_RESULTS_PREFIX: Gold_ResultsSets_PointOfTruth
OUTPUT_ROOT: F:\RC13_RC2023_24_Gold\report_card_bundle_notebook
REGIONS: CY,WT,MW,FI,BU,BM
CONSTITUENTS: FS,PN,PP,DIN,DON,DIP,DOP
UNITS: report
PROCESS_MODEL: BASE


## 3. Helper function to run commands

In [3]:
# ------------------------------------------------------------------------------
# Command helper functions
# ------------------------------------------------------------------------------
#
# These helper functions let the notebook call the same production CLI used in
# PowerShell:
#
#     python scripts/run_report_card_summary.py ...
#
# This is better than importing internal package functions directly because the
# notebook follows the same entry point as operational runs.
# ------------------------------------------------------------------------------

def run_cmd(cmd: list[str], *, check: bool = True):
    """
    Run a report-card CLI command from the repository root.

    Parameters
    ----------
    cmd : list[str]
        Complete command to run. The first item should normally be sys.executable
        so the notebook uses the same Python environment as the selected kernel.

    check : bool, default=True
        If True, raise an error when the command fails.
        If False, print stdout/stderr and continue so the error can be inspected.

    Why this function is needed
    ---------------------------
    The report-card runner writes useful information to stdout and stderr.
    Capturing and printing both streams makes debugging easier, especially when
    argparse fails or a workflow raises an exception.

    Returns
    -------
    subprocess.CompletedProcess
        The completed process object, including returncode, stdout and stderr.
    """
    print("Running command:")
    print(" ".join(f'"{x}"' if " " in str(x) else str(x) for x in cmd))
    print()

    result = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        check=False,
    )

    print("Return code:", result.returncode)

    print("\nSTDOUT:")
    print(result.stdout)

    print("\nSTDERR:")
    print(result.stderr)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with return code {result.returncode}"
        )

    return result


def base_cmd(output_root: str | Path = OUTPUT_ROOT) -> list[str]:
    """
    Build the common report-card command.

    Parameters
    ----------
    output_root : str | Path
        Destination folder for the outputs from this run.

    Why this function is needed
    ---------------------------
    Most examples use the same core arguments:

        --main-path
        --model-results-prefix
        --output-root
        --regions
        --constituents
        --units
        --process-model

    This function defines those common arguments once. Each example then appends
    only the skip flags needed for that specific run.

    Examples
    --------
    Full run:
        cmd = base_cmd()

    Change plots only:
        cmd = base_cmd() + ["--skip-region", "--skip-basin", ...]

    RSDR only:
        cmd = base_cmd() + ["--skip-change-plots", "--skip-modelled-vs-measured", ...]
    """
    return [
        sys.executable,
        str(SCRIPT),
        "--main-path", MAIN_PATH,
        "--model-results-prefix", MODEL_RESULTS_PREFIX,
        "--output-root", str(output_root),
        "--regions", REGIONS,
        "--constituents", CONSTITUENTS,
        "--units", UNITS,
        "--process-model", PROCESS_MODEL,
    ]

## 4. Full report-card bundle

This runs the complete bundle including summary tables, region/basin/subcatchment outputs, land use, source-sink, Sankey, FU load, change plots, modelled-vs-measured, and RSDR outputs.

In [4]:
RUN_FULL_BUNDLE = True

if RUN_FULL_BUNDLE:
    cmd = base_cmd(
        r"F:\RC13_RC2023_24_Gold\report_card_bundle_full_notebook"
    )

    run_cmd(cmd)
else:
    print("Skipped. Set RUN_FULL_BUNDLE = True to run.")

Running command:
"c:\Users\lius\OneDrive - ITP (Queensland Government)\ShuciL\CY_rebuild\Code\GBR_source_summary\.venv\Scripts\python.exe" "C:\Users\lius\OneDrive - ITP (Queensland Government)\ShuciL\CY_rebuild\Code\GBR_source_summary\scripts\run_report_card_summary.py" --main-path F:\RC13_RC2023_24_Gold --model-results-prefix Gold_ResultsSets_PointOfTruth --output-root F:\RC13_RC2023_24_Gold\report_card_bundle_full_notebook --regions CY,WT,MW,FI,BU,BM --constituents FS,PN,PP,DIN,DON,DIP,DOP --units report --process-model BASE

Return code: 0

STDOUT:
Inferred model years: 30
GBRCLMP file: C:\Users\lius\OneDrive - ITP (Queensland Government)\ShuciL\CY_rebuild\Code\GBR_source_summary\src\gbr_source_summary\data\monitoring_data\flowsLoadsGBRCLMP_GBR_allSites_MASTER.csv
Model results prefix: F:\RC13_RC2023_24_Gold\Gold_ResultsSets_PointOfTruth

REPORT-CARD BUNDLE STEP ORDER
[01/13] report_card_summary
[02/13] subcatchment_summary
[03/13] rsdr_shapefiles
[04/13] process_contribution_plots


## 5. Fast summary run

This skips heavier plotting/spatial/modelled-vs-measured outputs and keeps the main report-card summaries.

In [5]:
RUN_FAST_SUMMARY = True

if RUN_FAST_SUMMARY:
    cmd = base_cmd(
        r"F:\RC13_RC2023_24_Gold\report_card_bundle_fast_summary"
    ) + [
        "--skip-sankey",
        "--skip-process-plots",
        "--skip-change-plots",
        "--skip-modelled-vs-measured",
        "--skip-modelled-vs-measured-plots",
        "--skip-rsdr",
    ]

    run_cmd(cmd)
else:
    print("Skipped. Set RUN_FAST_SUMMARY = True to run.")

Running command:
"c:\Users\lius\OneDrive - ITP (Queensland Government)\ShuciL\CY_rebuild\Code\GBR_source_summary\.venv\Scripts\python.exe" "C:\Users\lius\OneDrive - ITP (Queensland Government)\ShuciL\CY_rebuild\Code\GBR_source_summary\scripts\run_report_card_summary.py" --main-path F:\RC13_RC2023_24_Gold --model-results-prefix Gold_ResultsSets_PointOfTruth --output-root F:\RC13_RC2023_24_Gold\report_card_bundle_fast_summary --regions CY,WT,MW,FI,BU,BM --constituents FS,PN,PP,DIN,DON,DIP,DOP --units report --process-model BASE --skip-sankey --skip-process-plots --skip-change-plots --skip-modelled-vs-measured --skip-modelled-vs-measured-plots --skip-rsdr

Return code: 0

STDOUT:
Inferred model years: 30

REPORT-CARD BUNDLE STEP ORDER
[01/08] report_card_summary
[02/08] subcatchment_summary
[03/08] region_summary
[04/08] basin_summary
[05/08] subcatchment_summary
[06/08] landuse_rainfall
[07/08] source_sink_summary
[08/08] fu_load_summary

[01/08] START: report_card_summary
--------------

## 6. Change summary plots only

Use this while developing or checking `workflows_change_summary_plots.py`.

This skips most products and only keeps what is needed to generate the change summary plots.

In [ ]:
RUN_CHANGE_PLOTS_ONLY = True

if RUN_CHANGE_PLOTS_ONLY:
    cmd = base_cmd(
        r"F:\RC13_RC2023_24_Gold\report_card_bundle_change_plots_only"
    ) + [
        "--skip-region",
        "--skip-basin",
        "--skip-subcatchment",
        "--skip-landuse",
        "--skip-source-sink",
        "--skip-sankey",
        "--skip-fu-load",
        "--skip-process-plots",
        "--skip-modelled-vs-measured",
        "--skip-modelled-vs-measured-plots",
        "--skip-rsdr",
        "--no-excel",
        "--fail-fast",
    ]

    run_cmd(cmd)
else:
    print("Skipped. Set RUN_CHANGE_PLOTS_ONLY = True to run.")

Running command:
"c:\Users\lius\OneDrive - ITP (Queensland Government)\ShuciL\CY_rebuild\Code\GBR_source_summary\.venv\Scripts\python.exe" "C:\Users\lius\OneDrive - ITP (Queensland Government)\ShuciL\CY_rebuild\Code\GBR_source_summary\scripts\run_report_card_summary.py" --main-path F:\RC13_RC2023_24_Gold --model-results-prefix Gold_ResultsSets_PointOfTruth --output-root F:\RC13_RC2023_24_Gold\report_card_bundle_change_plots_only --regions CY,WT,MW,FI,BU,BM --constituents FS,PN,PP,DIN,DON,DIP,DOP --units report --process-model BASE --skip-region --skip-basin --skip-subcatchment --skip-landuse --skip-source-sink --skip-sankey --skip-fu-load --skip-process-plots --skip-modelled-vs-measured --skip-modelled-vs-measured-plots --skip-rsdr --no-excel --fail-fast



## 7. RSDR exports only

Use this to generate the RSDR CSV/shapefile outputs using the packaged regional spatial layers.

In [ ]:
RUN_RSDR_ONLY = True

if RUN_RSDR_ONLY:
    cmd = base_cmd(
        r"F:\RC13_RC2023_24_Gold\report_card_bundle_rsdr_only"
    ) + [
        "--skip-region",
        "--skip-basin",
        "--skip-subcatchment",
        "--skip-landuse",
        "--skip-source-sink",
        "--skip-sankey",
        "--skip-fu-load",
        "--skip-process-plots",
        "--skip-change-plots",
        "--skip-modelled-vs-measured",
        "--skip-modelled-vs-measured-plots",
        "--no-excel",
        "--fail-fast",
    ]

    run_cmd(cmd)
else:
    print("Skipped. Set RUN_RSDR_ONLY = True to run.")

## 8. Modelled-vs-measured / Moriasi only

This runs the modelled-vs-measured workflow and Moriasi statistics.

The monitoring master CSV is taken from the packaged data unless overridden in the CLI.

In [ ]:
RUN_MODELLED_VS_MEASURED_ONLY = True

if RUN_MODELLED_VS_MEASURED_ONLY:
    cmd = base_cmd(
        r"F:\RC13_RC2023_24_Gold\report_card_bundle_modelled_vs_measured_only"
    ) + [
        "--skip-region",
        "--skip-basin",
        "--skip-subcatchment",
        "--skip-landuse",
        "--skip-source-sink",
        "--skip-sankey",
        "--skip-fu-load",
        "--skip-process-plots",
        "--skip-change-plots",
        "--skip-rsdr",
        "--no-excel",
        "--fail-fast",
    ]

    run_cmd(cmd)
else:
    print("Skipped. Set RUN_MODELLED_VS_MEASURED_ONLY = True to run.")

## 9. Modelled-vs-measured tables only, no plots

This keeps Moriasi/modelled-vs-measured tables but skips the PNG plots.

In [ ]:
RUN_MODELLED_VS_MEASURED_TABLES_ONLY = True

if RUN_MODELLED_VS_MEASURED_TABLES_ONLY:
    cmd = base_cmd(
        r"F:\RC13_RC2023_24_Gold\report_card_bundle_mvm_tables_only"
    ) + [
        "--skip-region",
        "--skip-basin",
        "--skip-subcatchment",
        "--skip-landuse",
        "--skip-source-sink",
        "--skip-sankey",
        "--skip-fu-load",
        "--skip-process-plots",
        "--skip-change-plots",
        "--skip-rsdr",
        "--skip-modelled-vs-measured-plots",
        "--no-excel",
        "--fail-fast",
    ]

    run_cmd(cmd)
else:
    print("Skipped. Set RUN_MODELLED_VS_MEASURED_TABLES_ONLY = True to run.")

## 10. Debugging run with `--fail-fast`

Use this when a workflow fails and you want the full traceback immediately.

In [ ]:
RUN_DEBUG_FAIL_FAST = True

if RUN_DEBUG_FAIL_FAST:
    cmd = base_cmd(
        r"F:\RC13_RC2023_24_Gold\report_card_bundle_debug"
    ) + [
        "--skip-sankey",
        "--skip-modelled-vs-measured-plots",
        "--fail-fast",
    ]

    run_cmd(cmd)
else:
    print("Skipped. Set RUN_DEBUG_FAIL_FAST = True to run.")

## 11. Example: one-region test run

Useful for quickly testing code changes on one region before running all six regions.

In [ ]:
RUN_ONE_REGION_TEST = True

if RUN_ONE_REGION_TEST:
    cmd = [
        sys.executable,
        str(SCRIPT),
        "--main-path", MAIN_PATH,
        "--model-results-prefix", MODEL_RESULTS_PREFIX,
        "--output-root", r"F:\RC13_RC2023_24_Gold\report_card_bundle_BM_test",
        "--regions", "BM",
        "--constituents", CONSTITUENTS,
        "--units", UNITS,
        "--process-model", PROCESS_MODEL,
        "--skip-sankey",
        "--skip-modelled-vs-measured",
        "--fail-fast",
    ]

    run_cmd(cmd)
else:
    print("Skipped. Set RUN_ONE_REGION_TEST = True to run.")

## 12. Output folder check

In [ ]:
out = Path(OUTPUT_ROOT)

if out.exists():
    for p in sorted(out.glob("*")):
        print(p)
else:
    print("Output folder does not exist yet:", out)

## 13. Notes

### Main path

`--main-path` should point to the model/report-card root folder, for example:

```text
F:\RC13_RC2023_24_Gold
```

### Model results prefix

`--model-results-prefix` should point to the folder immediately above the region folders. It can be relative to `--main-path`:

```text
Gold_ResultsSets_PointOfTruth
```

This resolves to:

```text
F:\RC13_RC2023_24_Gold\Gold_ResultsSets_PointOfTruth
```

### Packaged data

The package includes:

```text
src/gbr_source_summary/data/monitoring_data/
src/gbr_source_summary/data/region_lookup/
src/gbr_source_summary/data/spatial/regions_shp/
```


```